# OCEAN LoRA-combination experiment — starter analysis

Downloads the per-config TRAIT + MMLU results from HF, aggregates each config to a
mean value with **ci95 error bars** (Wilson for the binary MMLU accuracy, BCa
bootstrap for the continuous trait scores — per the repo convention), and plots
the 32 configs against **Σscale** (total adapter magnitude) on the x-axis.

Starter scaffolding — extend with the regressions / groupings you actually want.

In [ ]:
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from dotenv import load_dotenv
from huggingface_hub.errors import EntryNotFoundError

from scripts_dev.combinations_experiment.config_design import TRAITS, generate_design
from scripts_dev.combinations_experiment.run_experiment import HF_PREFIX, HF_REPO_ID
from src_dev.utils.hf_hub import download_path_to_dir
# CLAUDE.md sanctions using these interval helpers directly; reuse the per-sample
# parser rather than reimplementing its four scoring conventions.
from src_dev.evals.personality.analyze_results import (
    _extract_raw_sample_scores,
    _interval_ci_from_bootstrap,
    _interval_ci_from_wilson,
)

load_dotenv()

CACHE_ROOT = Path("scratch/combinations_experiment_analysis/cache")
CONFIDENCE = 95
BOOT_RESAMPLES = 1000
BOOT_SEED = 42
DOWNLOAD = True  # set False to reuse a previously downloaded cache

In [ ]:
def _log_json(config_dir: Path, eval_name: str) -> Path | None:
    # Latest log (timestamped name) — defensive against any stale earlier log.
    logs = sorted((config_dir / eval_name / "native" / "inspect_logs").glob("*.json"))
    return logs[-1] if logs else None


def _mmlu_mean_ci(log_path: Path) -> tuple[float, float, float] | None:
    """MMLU accuracy: mean + Wilson ci95 over the binary 0/1 per-sample scores."""
    raw = _extract_raw_sample_scores(log_path, "mmlu")
    if not raw or "accuracy" not in raw:
        return None
    vals = np.asarray(raw["accuracy"], dtype=float)
    lo, hi = _interval_ci_from_wilson(vals, CONFIDENCE)
    return float(vals.mean()), lo, hi


def _trait_means_ci(log_path: Path) -> dict[str, tuple[float, float, float]]:
    """Per-trait score: mean + bootstrap ci95 over the continuous per-sample scores."""
    raw = _extract_raw_sample_scores(log_path, "trait_logprobs") or {}
    out: dict[str, tuple[float, float, float]] = {}
    for key, scores in raw.items():
        if key.startswith("_"):
            continue  # skip helper keys (_cm_*, _nc_*, _choice_mass)
        letter = key[0].upper()  # Openness->O, Conscientiousness->C, ...
        vals = np.asarray(scores, dtype=float)
        lo, hi = _interval_ci_from_bootstrap(vals, CONFIDENCE, BOOT_RESAMPLES, BOOT_SEED)
        out[letter] = (float(vals.mean()), lo, hi)
    return out

In [ ]:
design = generate_design()
rows = []
missing = []

for record in design.configs:
    config_dir = CACHE_ROOT / record.slug
    if DOWNLOAD:
        try:
            download_path_to_dir(
                repo_id=HF_REPO_ID,
                path_in_repo=f"{HF_PREFIX}/{record.slug}",
                target_dir=config_dir,
                allow_patterns=["**/*.json"],
            )
        except EntryNotFoundError:
            missing.append(record.slug)  # not uploaded yet (run still in progress)
            continue
    if not config_dir.exists():
        missing.append(record.slug)
        continue

    row = {"slug": record.slug, "sumscale": record.sumscale_actual}
    for t in record.traits:
        row[f"dir_{t.letter}"] = t.direction

    mmlu_log = _log_json(config_dir, "mmlu")
    if mmlu_log is not None and (res := _mmlu_mean_ci(mmlu_log)) is not None:
        row["mmlu_mean"], row["mmlu_lo"], row["mmlu_hi"] = res

    trait_log = _log_json(config_dir, "trait_logprobs")
    if trait_log is not None:
        for letter, (mean, lo, hi) in _trait_means_ci(trait_log).items():
            row[f"trait_{letter}_mean"] = mean
            row[f"trait_{letter}_lo"] = lo
            row[f"trait_{letter}_hi"] = hi

    rows.append(row)

df = pd.DataFrame(rows)
if not df.empty:
    df = df.sort_values("sumscale").reset_index(drop=True)
n_mmlu = int(df["mmlu_mean"].notna().sum()) if "mmlu_mean" in df.columns else 0
print(f"{len(df)} configs loaded; {len(missing)} not on HF yet; {n_mmlu} with MMLU results")
df

In [ ]:
# MMLU accuracy vs Σscale (32 points, Wilson ci95 error bars).
if "mmlu_mean" not in df.columns or df["mmlu_mean"].notna().sum() == 0:
    print("No MMLU results yet — skipping plot.")
else:
    d = df.dropna(subset=["mmlu_mean"])
    yerr = np.vstack([d["mmlu_mean"] - d["mmlu_lo"], d["mmlu_hi"] - d["mmlu_mean"]])

    fig, ax = plt.subplots(figsize=(7, 4.5))
    ax.errorbar(d["sumscale"], d["mmlu_mean"], yerr=yerr, fmt="o", capsize=3, color="tab:blue")
    ax.set_xlabel("Σ scale (total adapter magnitude)")
    ax.set_ylabel("MMLU accuracy")
    ax.set_title("MMLU vs total adapter magnitude (Wilson ci95)")
    ax.grid(True, alpha=0.3)
    fig.tight_layout()
    plt.show()

In [ ]:
# Per-trait score vs Σscale (32 points each, bootstrap ci95), coloured by direction.
colors = {"P": "tab:green", "M": "tab:red"}
fig, axes = plt.subplots(1, len(TRAITS), figsize=(4 * len(TRAITS), 3.6), sharey=True)

for ax, letter in zip(axes, TRAITS):
    mean_col = f"trait_{letter}_mean"
    if mean_col not in df:
        ax.set_visible(False)
        continue
    d = df.dropna(subset=[mean_col])
    yerr = np.vstack([d[mean_col] - d[f"trait_{letter}_lo"], d[f"trait_{letter}_hi"] - d[mean_col]])
    point_colors = [colors[v] for v in d[f"dir_{letter}"]]
    ax.errorbar(d["sumscale"], d[mean_col], yerr=yerr, fmt="none", ecolor="lightgray", zorder=1)
    ax.scatter(d["sumscale"], d[mean_col], c=point_colors, zorder=2)
    ax.set_xlabel("Σ scale")
    ax.set_title(letter)
    ax.grid(True, alpha=0.3)

axes[0].set_ylabel("trait score")
fig.suptitle("Per-trait score vs Σscale  (green = amplifier, red = suppressor)")
fig.tight_layout()
plt.show()